phase 3 Regularization in ML

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso

In [2]:
df = pd.read_csv("50_Startups.csv")
df.head()

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   R&D Spend        50 non-null     float64
 1   Administration   50 non-null     float64
 2   Marketing Spend  50 non-null     float64
 3   State            50 non-null     str    
 4   Profit           50 non-null     float64
dtypes: float64(4), str(1)
memory usage: 2.5 KB


In [4]:
df=pd.get_dummies(df,columns=["State"],drop_first=True)
df.head()

,R&D Spend,Administration,Marketing Spend,Profit,State_Florida,State_New York
0,165349.20,136897.80,471784.10,192261.83,False,True
1,162597.70,151377.59,443898.53,191792.06,False,False
2,153441.51,101145.55,407934.54,191050.39,True,False
3,144372.41,118671.85,383199.62,182901.99,False,True
4,142107.34,91391.77,366168.42,166187.94,True,False


In [5]:
X = df.drop("Profit", axis=1)
y = df["Profit"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [7]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [8]:

models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1.0),
}

for name, model in models.items():
    model.fit(X_train_s, y_train)
    print(name,
          "| Train R2:", round(model.score(X_train_s, y_train), 4),
          "| Test R2:", round(model.score(X_test_s, y_test), 4))

Linear | Train R2: 0.9537 | Test R2: 0.8987
Ridge | Train R2: 0.9527 | Test R2: 0.8954
Lasso | Train R2: 0.9537 | Test R2: 0.8987


In [9]:
for alpha in [0.01, 0.1, 1, 10, 100]:
    r = Ridge(alpha=alpha).fit(X_train_s, y_train)
    l = Lasso(alpha=alpha).fit(X_train_s, y_train)
    print(alpha,
          "| Ridge test:", round(r.score(X_test_s, y_test), 4),
          "| Lasso test:", round(l.score(X_test_s, y_test), 4))

0.01 | Ridge test: 0.8987 | Lasso test: 0.8987
0.1 | Ridge test: 0.8985 | Lasso test: 0.8987
1 | Ridge test: 0.8954 | Lasso test: 0.8987
10 | Ridge test: 0.8566 | Lasso test: 0.8989
100 | Ridge test: 0.5377 | Lasso test: 0.9008


In [10]:
lasso = Lasso(alpha=10)
lasso.fit(X_train_s, y_train)

coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Lasso coef": lasso.coef_
})
print(coef_df)

           Feature    Lasso coef
0        R&D Spend  38094.411719
1   Administration  -1853.913172
2  Marketing Spend   3385.455315
3    State_Florida    436.989864
4   State_New York     -0.000000


In [11]:
from sklearn.linear_model import RidgeCV, LassoCV

alphas = [0.01, 0.1, 1, 10, 100]

ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train_s, y_train)

lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=42)
lasso_cv.fit(X_train_s, y_train)

print("Best Ridge alpha:", ridge_cv.alpha_)
print("Best Lasso alpha:", lasso_cv.alpha_)
print("Ridge test R2:", round(ridge_cv.score(X_test_s, y_test), 4))
print("Lasso test R2:", round(lasso_cv.score(X_test_s, y_test), 4))

Best Ridge alpha: 0.1
Best Lasso alpha: 100.0
Ridge test R2: 0.8985
Lasso test R2: 0.9008
